# 02 – Nettoyage des données

Objectif : produire une version propre du jeu de données, utilisable pour les agrégations.
Chaque décision s'appuie sur un constat de `01_exploration.ipynb` et est documentée
(quoi / pourquoi / nombre de lignes touchées).

Entrée : `data/raw/Accidental_Drug_Related_Deaths_2012-2022.csv` (jamais modifié)
Sortie : `data/processed/` (fichier nettoyé)

In [1]:
import pandas as pd

import sys
sys.path.append("..")
from src.nettoyage import convertir_en_booleens

deces = pd.read_csv("../data/raw/Accidental_Drug_Related_Deaths_2012-2022.csv")
nb_lignes_depart = len(deces)
print(f"Lignes au départ : {nb_lignes_depart}, colonnes : {deces.shape[1]}")

Lignes au départ : 10654, colonnes : 48


**Décision 1 – Suppression des colonnes écartées**
- Quoi : suppression de `Other`, `Other Opioid` et `Ethnicity` (3 colonnes, aucune ligne).
- Pourquoi : `Other` = 147 valeurs en texte libre, même substance sous plusieurs écritures (constat 12) ;
  `Other Opioid` = 39 noms de médicaments, dont des non-opioïdes (constat 4) ;
  `Ethnicity` = 85 % de vides, non utilisée par nos questions (constat 2).
- Les autres colonnes en texte libre sont gardées car elles ne gênent pas l'analyse.

In [2]:
colonnes_ecartees = ["Other", "Other Opioid", "Ethnicity"]
deces = deces.drop(columns=colonnes_ecartees)
print(f"Colonnes après suppression : {deces.shape[1]}")

Colonnes après suppression : 45


**Décision 2 – Conversion des substances en booléens**
- Quoi : les colonnes de `Heroin` à `Any Opioid` passent en True (détectée) / False (non détectée).
- Pourquoi : pour compter les détections avec `sum`. Valeurs considérées comme détectées :
  `Y`, `y`, `Y POPS`, `Y (PTCH)`, soit 48 détections qu'un simple `== "Y"` aurait perdues.
  Tout le reste est considéré comme non détecté : vides, `N` (8 dans `Any Opioid`),
  `P` (1 dans `Ethanol`, ambigu) et les 3 notes en texte libre de `Morphine (Not Heroin)` (constat 4).

In [3]:
colonnes_substances = deces.loc[:, "Heroin":"Any Opioid"].columns
print(f"Nombre de colonnes de substances : {len(colonnes_substances)}")
deces = convertir_en_booleens(deces, colonnes_substances)
print(deces[colonnes_substances].dtypes.value_counts())

Nombre de colonnes de substances : 21
bool    21
Name: count, dtype: int64
